In [1]:
# Step 0 setup: find the project root and put src/ on the import path
import sys, importlib, inspect
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "src").is_dir():
    if root == root.parent:
        raise FileNotFoundError("Could not find the project's src/ folder")
    root = root.parent
sys.path.insert(0, str(root / "src"))

import config, features
importlib.reload(config)
importlib.reload(features)
from config import TRAIN_PATH, VALID_PATH, DECEMBER_PATH, DATE_COL

In [2]:
# Check A: features.py no longer defines DATE_COL itself
src = inspect.getsource(features)
print("features defines DATE_COL itself:", 'DATE_COL = "date"' in src)   # expect False
print("features.DATE_COL =", repr(features.DATE_COL))                    # expect 'date'

# Check B: every distance in all three files gets a band (raw files; cleaning never changes distance)
files = {"train": pd.read_csv(TRAIN_PATH),
         "validation": pd.read_csv(VALID_PATH),
         "december": pd.read_csv(DECEMBER_PATH)}
for name, df in files.items():
    bands = features.distance_band(df["distance"])   # raises if any band is NaN
    print(f"\n{name}: rows={len(df):,}  distance min={df['distance'].min():.1f}  max={df['distance'].max():.1f}")
    print(bands.value_counts(sort=False).to_string())

# Check C: the safety net actually fires on bad input
try:
    features.distance_band(pd.Series([100.0, 4500.0, None]))
except ValueError as e:
    print("\nSafety check works:", e)
else:
    print("\nPROBLEM: no error raised")

features defines DATE_COL itself: False
features.DATE_COL = 'date'

train: rows=48,000  distance min=70.0  max=3439.8
distance
(0, 250]         2713
(250, 500]       7577
(500, 1000]     15055
(1000, 2000]    15108
(2000, 4000]     7547

validation: rows=12,000  distance min=70.0  max=3325.7
distance
(0, 250]         683
(250, 500]      1881
(500, 1000]     3735
(1000, 2000]    3792
(2000, 4000]    1909

december: rows=31  distance min=360.0  max=360.0
distance
(0, 250]         0
(250, 500]      31
(500, 1000]      0
(1000, 2000]     0
(2000, 4000]     0

Safety check works: 2 distance(s) missing or outside 0-4000 mi, e.g. [4500.0, nan]


In [3]:
# ===== Step 1: time splits =====
import validation
importlib.reload(validation)

raw_train = pd.read_csv(TRAIN_PATH)
splits = {}
for fold in [validation.DECISION_FOLD, validation.MAIN_FOLD]:
    tr, te = validation.time_split(raw_train, fold)
    splits[fold.name] = (tr, te)
    print(validation.split_summary(tr, te, fold).to_string(index=False), "\n")

dec_tr, dec_te = splits["decision"]
main_tr, main_te = splits["main"]

# Cross-checks
print("main covers every row (train+test = 48,000):", len(main_tr) + len(main_te) == len(raw_train))
print("Aug + Sep-Oct = 14,282 (Step 13's Aug-Oct holdout):", len(dec_te) + len(main_te))
dec_dates = pd.to_datetime(pd.concat([dec_tr, dec_te])[DATE_COL])
print("decision fold touches Sep-Oct:", (dec_dates >= "2025-09-01").any())   # expect False
print("decision test ids inside main TRAIN:", set(dec_te["load_id"]) <= set(main_tr["load_id"]))  # expect True
print("raw_train date still text (untouched):", raw_train[DATE_COL].dtype)    # expect object

# Safety check: an overlapping fold must be refused
bad = validation.Fold("bad", "2025-01-01", "2025-08-31", "2025-08-15", "2025-10-31")
try:
    validation.time_split(raw_train, bad)
    print("PROBLEM: overlapping fold was NOT refused")
except ValueError as e:
    print("overlap refused as expected:", e)

    fold  side  rows      first       last
decision train 33718 2025-01-01 2025-07-31
decision  test  4759 2025-08-01 2025-08-31 

fold  side  rows      first       last
main train 38477 2025-01-01 2025-08-31
main  test  9523 2025-09-01 2025-10-31 

main covers every row (train+test = 48,000): True
Aug + Sep-Oct = 14,282 (Step 13's Aug-Oct holdout): 14282
decision fold touches Sep-Oct: False
decision test ids inside main TRAIN: True
raw_train date still text (untouched): object
overlap refused as expected: Fold 'bad': training must end before testing starts


In [4]:
# ===== Step 2: metrics =====
importlib.reload(validation)
import numpy as np

# Toy 1: compare with your hand calculation
true1, pred1 = [100, 200, 400], [110, 180, 400]
print("Toy 1:", {k: round(v, 3) for k, v in validation.score(true1, pred1).items()})

# Toy 2: one big miss among four loads. How do MAE and RMSE react?
true2, pred2 = [100, 100, 100, 100], [100, 100, 100, 200]
print("Toy 2:", {k: round(v, 3) for k, v in validation.score(true2, pred2).items()})

# Toy 3: a label error with a falsely LOW true price. How does MAPE react?
true3, pred3 = [200, 200, 50], [200, 200, 200]
print("Toy 3:", {k: round(v, 3) for k, v in validation.score(true3, pred3).items()})

# Cross-check against scikit-learn on 1,000 random rates
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
rng = np.random.default_rng(42)
t = rng.uniform(200, 5000, 1000)
p = t * rng.uniform(0.7, 1.3, 1000)
print("\nmatches sklearn MAE: ", np.isclose(validation.mae(t, p), mean_absolute_error(t, p)))
print("matches sklearn MAPE:", np.isclose(validation.mape(t, p), mean_absolute_percentage_error(t, p) * 100))
print("matches sklearn RMSE:", np.isclose(validation.rmse(t, p), root_mean_squared_error(t, p)))

# Safety checks: bad inputs must stop the run
for label, (tt, pp) in {"zero true rate": ([0, 100], [50, 100]),
                        "NaN prediction": ([100, 100], [np.nan, 100]),
                        "length mismatch": ([100, 100], [100])}.items():
    try:
        validation.score(tt, pp)
        print(f"PROBLEM: {label} NOT caught")
    except ValueError as e:
        print(f"{label} caught: {e}")

Toy 1: {'MAE': 10.0, 'MAPE_%': 6.667, 'RMSE': 12.91, 'bias': -3.333}
Toy 2: {'MAE': 25.0, 'MAPE_%': 25.0, 'RMSE': 50.0, 'bias': 25.0}
Toy 3: {'MAE': 50.0, 'MAPE_%': 100.0, 'RMSE': 86.603, 'bias': 50.0}

matches sklearn MAE:  True
matches sklearn MAPE: True
matches sklearn RMSE: True
zero true rate caught: True rates must be > 0 (MAPE divides by them)
NaN prediction caught: NaN or infinite values in true or predicted rates
length mismatch caught: Length mismatch: (2,) true vs (1,) predicted


In [5]:
# ===== Step 3a: full-file reproduction of the 677 =====
importlib.reload(config); importlib.reload(features)   # config first: features imports from it
import baselines; importlib.reload(baselines)

learned_full = features.fit_cleaning(raw_train)
clean_train = features.clean(raw_train, learned_full)

table_full = baselines.fit_price_table(clean_train)
sizes = baselines.rpm(clean_train).groupby(baselines._segment_keys(clean_train)).size()
sizes.index.names = table_full.index.names
print(pd.DataFrame({"median_rpm": table_full.round(3), "rows": sizes}).to_string())

flags_full = baselines.flag_label_errors(clean_train, table_full)
full_ids = set(clean_train.loc[flags_full["is_error"].to_numpy(), "load_id"])
print(f"\nlow: {flags_full['low'].sum()}  high: {flags_full['high'].sum()}  total: {flags_full['is_error'].sum()}")

                        median_rpm  rows
equipment band                          
Dry Van   (0, 250]           2.635  1559
          (1000, 2000]       1.968  8612
          (2000, 4000]       1.841  4299
          (250, 500]         2.325  4275
          (500, 1000]        2.108  8457
Flatbed   (0, 250]           2.850   481
          (1000, 2000]       2.129  2684
          (2000, 4000]       1.990  1380
          (250, 500]         2.521  1429
          (500, 1000]        2.280  2779
Reefer    (0, 250]           2.962   673
          (1000, 2000]       2.221  3812
          (2000, 4000]       2.074  1868
          (250, 500]         2.631  1873
          (500, 1000]        2.379  3819

low: 337  high: 340  total: 677


In [6]:
# ===== Step 3b: label-error flags per training fold =====
for fold in [validation.DECISION_FOLD, validation.MAIN_FOLD]:
    tr_raw, _ = validation.time_split(raw_train, fold)
    tr = features.clean(tr_raw, features.fit_cleaning(tr_raw))     # cleaning learned from the fold
    kept, flags = baselines.drop_label_errors(tr)                   # table learned from the fold
    ids = set(tr.loc[flags["is_error"].to_numpy(), "load_id"])
    print(f"{fold.name}: train {len(tr):,}  flagged {len(ids)} "
          f"(low {flags['low'].sum()}, high {flags['high'].sum()})  kept {len(kept):,}  | "
          f"full-file flags in this window: {len(full_ids & set(tr['load_id']))}  "
          f"| shared with them: {len(ids & full_ids)}")

decision: train 33,718  flagged 465 (low 231, high 234)  kept 33,253  | full-file flags in this window: 465  | shared with them: 465
main: train 38,477  flagged 533 (low 266, high 267)  kept 37,944  | full-file flags in this window: 533  | shared with them: 533


In [7]:
# ===== Step 3c: baseline mechanics (NOT scores: predicting the rows they learned from) =====
b0 = baselines.GlobalMedianRate().fit(clean_train)
b1 = baselines.SegmentMedianRpm().fit(clean_train)
for m in (b0, b1):
    p = m.predict(clean_train)
    print(f"{m.name}: n={len(p):,}  NaN={np.isnan(p).sum()}  "
          f"min={p.min():,.0f}  max={p.max():,.0f}  all>0={bool((p > 0).all())}")

# December sanity: B1 ignores the date, so expect ONE value for all 31 days.
# Phase 2 sanity band: 750-950.
clean_valid = features.clean(pd.read_csv(VALID_PATH), learned_full)
dec_clean = features.clean(pd.read_csv(DECEMBER_PATH), learned_full,
                           market_by_date=features.daily_market_index(clean_valid))
dec_p = b1.predict(dec_clean)
print(f"\nB1 December: unique values {np.unique(dec_p.round(2))}  "
      f"inside 750-950: {bool(((dec_p >= 750) & (dec_p <= 950)).all())}")

# Safety: a segment missing from the table must stop the run
try:
    baselines.lookup_rpm(clean_train, table_full.drop(("Reefer", "(0, 250]")))
    print("PROBLEM: missing table cell NOT caught")
except ValueError as e:
    print("missing table cell caught:", str(e)[:100])

B0 global median rate: n=48,000  NaN=0  min=2,031  max=2,031  all>0=True
B1 median rpm by equipment x band: n=48,000  NaN=0  min=184  max=7,135  all>0=True

B1 December: unique values [836.89]  inside 750-950: True
missing table cell caught: No price-table entry for: [('Reefer', '(0, 250]')]


In [8]:
# ===== Step 4: run_fold end to end =====
importlib.reload(features); importlib.reload(baselines); importlib.reload(validation)

results = []
for fold in [validation.DECISION_FOLD, validation.MAIN_FOLD]:
    for make in [baselines.GlobalMedianRate, baselines.SegmentMedianRpm]:
        r = validation.run_fold(raw_train, fold, make)
        results.append(r)
        h = r["holdout"]
        print(f"[{r['fold']}] {r['model']}")
        print(f"  cleaning learned from {r['cleaning_rows']:,} rows, last train date {r['train_last_date']}")
        print(f"  dropped {r['dropped']} label errors -> fit on {r['fit_rows']:,}  |  test rows {r['test_rows']:,}")
        print(f"  preds: n={len(h):,}  NaN={h['pred'].isna().sum()}  min={h['pred'].min():,.0f}  "
              f"all>0={bool((h['pred'] > 0).all())}  | holdout rows flagged: {h['is_label_error'].sum()}")
        print("  scores:", {k: round(v, 2) for k, v in r["scores"].items()}, "\n")

# The caller's data must be untouched after all runs
print("raw_train untouched:", len(raw_train) == 48_000 and raw_train[DATE_COL].dtype == object)

# Safety: a model that returns a zero price must be stopped
class BrokenModel:
    name = "broken"
    def fit(self, train): return self
    def predict(self, df): return np.zeros(len(df))
try:
    validation.run_fold(raw_train, validation.DECISION_FOLD, BrokenModel)
    print("PROBLEM: bad predictions NOT caught")
except ValueError as e:
    print("bad predictions caught:", e)

[decision] B0 global median rate
  cleaning learned from 33,718 rows, last train date 2025-07-31
  dropped 465 label errors -> fit on 33,253  |  test rows 4,759
  preds: n=4,759  NaN=0  min=2,029  all>0=True  | holdout rows flagged: 68
  scores: {'MAE': 1111.37, 'MAPE_%': 70.98, 'RMSE': 1506.88, 'bias': -309.16} 

[decision] B1 median rpm by equipment x band
  cleaning learned from 33,718 rows, last train date 2025-07-31
  dropped 465 label errors -> fit on 33,253  |  test rows 4,759
  preds: n=4,759  NaN=0  min=184  all>0=True  | holdout rows flagged: 68
  scores: {'MAE': 136.6, 'MAPE_%': 6.39, 'RMSE': 621.99, 'bias': 9.68} 

[main] B0 global median rate
  cleaning learned from 38,477 rows, last train date 2025-08-31
  dropped 533 label errors -> fit on 37,944  |  test rows 9,523
  preds: n=9,523  NaN=0  min=2,028  all>0=True  | holdout rows flagged: 144
  scores: {'MAE': 1148.9, 'MAPE_%': 70.21, 'RMSE': 1569.0, 'bias': -364.6} 

[main] B1 median rpm by equipment x band
  cleaning lea

In [9]:
# ===== Step 5a: baseline scores table =====
importlib.reload(validation)
pd.set_option("display.width", 200)

# Comparison runs (D-6): label errors KEPT in training, decision fold only
keep_runs = [validation.run_fold(raw_train, validation.DECISION_FOLD, m, drop_errors=False)
             for m in (baselines.GlobalMedianRate, baselines.SegmentMedianRpm)]

scores_table = validation.results_table(results + keep_runs)    # `results` from Step 4
print(scores_table.round(2).to_string(index=False))

out = config.OUTPUT_DIR / "phase3_baseline_scores.csv"
out.parent.mkdir(exist_ok=True)
scores_table.round(4).to_csv(out, index=False)
print("\nsaved:", out.relative_to(config.PROJECT_ROOT))

    fold                             model train_label_errors      scope  rows     MAE  MAPE_%    RMSE    bias
decision             B0 global median rate           excluded        all  4759 1111.37   70.98 1506.88 -309.16
decision             B0 global median rate           excluded clean-only  4691 1074.45   68.39 1356.31 -283.19
decision B1 median rpm by equipment x band           excluded        all  4759  136.60    6.39  621.99    9.68
decision B1 median rpm by equipment x band           excluded clean-only  4691   87.25    3.98  122.94   38.14
    main             B0 global median rate           excluded        all  9523 1148.90   70.21 1569.00 -364.60
    main             B0 global median rate           excluded clean-only  9379 1107.30   67.77 1414.82 -332.30
    main B1 median rpm by equipment x band           excluded        all  9523  140.53    6.40  638.82  -43.12
    main B1 median rpm by equipment x band           excluded clean-only  9379   85.01    3.90  116.40   -9.55
d

In [10]:
# ===== Step 5b: B1 breakdowns on the main holdout =====
main_b1 = next(r for r in results if r["fold"] == "main" and r["model"].startswith("B1"))
h = main_b1["holdout"]

for by in ["equipment", "band", "month"]:
    print(f"\n--- main holdout, B1, by {by} ---")
    print(validation.breakdown(h, by).round(2).to_string(index=False))

# How much of B1's squared error comes from the flagged label-error rows?
sq_err = (h["pred"] - h[config.TARGET]) ** 2
share_rows = h["is_label_error"].mean()
share_sq = sq_err[h["is_label_error"]].sum() / sq_err.sum()
print(f"\nlabel-error rows: {share_rows:.1%} of holdout rows, {share_sq:.1%} of squared error")


--- main holdout, B1, by equipment ---
equipment  rows  label_errors    MAE  MAPE_%   RMSE   bias
  Dry Van  5360            85 139.26    6.20 659.89 -46.64
  Flatbed  1770            24 133.98    6.65 516.14 -37.61
   Reefer  2393            35 148.23    6.66 672.01 -39.30

--- main holdout, B1, by band ---
        band  rows  label_errors    MAE  MAPE_%   RMSE   bias
    (0, 250]   550             6  30.95    6.75 106.33  -6.12
  (250, 500]  1540            29  71.26    7.57 274.26 -26.82
 (500, 1000]  2972            45 102.61    6.79 371.47 -23.16
(1000, 2000]  2927            43 181.97    5.79 830.89 -64.57
(2000, 4000]  1534            21 243.77    5.51 932.31 -70.47

--- main holdout, B1, by month ---
  month  rows  label_errors    MAE  MAPE_%   RMSE   bias
2025-09  4670            64 136.27    5.91 622.35 -39.66
2025-10  4853            80 144.63    6.87 654.28 -46.45

label-error rows: 1.5% of holdout rows, 96.7% of squared error


In [11]:
# ===== Step 5c: is October worse because of drift, or just more label errors? =====
print(validation.breakdown(h[~h["is_label_error"]], "month").round(2).to_string(index=False))

  month  rows  label_errors   MAE  MAPE_%   RMSE   bias
2025-09  4606             0 85.33    3.88 117.84  -7.22
2025-10  4773             0 84.69    3.91 114.99 -11.80


In [12]:
# ===== Step 6a: regression check after the refactor =====
importlib.reload(validation)
metric_cols = ["MAE", "MAPE_%", "RMSE", "bias"]
recheck = validation.results_table(
    [validation.run_fold(raw_train, f, m)
     for f in [validation.DECISION_FOLD, validation.MAIN_FOLD]
     for m in [baselines.GlobalMedianRate, baselines.SegmentMedianRpm]])
saved = pd.read_csv(config.OUTPUT_DIR / "phase3_baseline_scores.csv").iloc[:8]   # first 8 = these runs
print("refactor changed nothing:",
      np.allclose(recheck[metric_cols].round(4).to_numpy(), saved[metric_cols].to_numpy()))

refactor changed nothing: True


In [13]:
# ===== Step 6b: random split vs time split =====
n_test = 9_523                                   # same size as the Sep-Oct holdout
random_runs = []
for seed in [42, 1, 2]:                          # 3 seeds: is any gap bigger than random noise?
    tr, te = validation.random_split(raw_train, n_test=n_test, seed=seed)
    if seed == 42:
        months = pd.to_datetime(te[DATE_COL]).dt.month
        print(f"seed 42: train {len(tr):,} rows, last train date {pd.to_datetime(tr[DATE_COL]).max().date()}; "
              f"test drawn from {months.nunique()} months, {(months >= 9).mean():.1%} from Sep-Oct\n")
    for m in [baselines.GlobalMedianRate, baselines.SegmentMedianRpm]:
        random_runs.append(validation.run_split(tr, te, f"random seed {seed}", m))

main_runs = [r for r in results if r["fold"] == "main"]          # time split, from Step 4
cmp = validation.results_table(main_runs + random_runs)
cols = ["fold", "model", "scope", "rows"] + metric_cols
for scope in ["all", "clean-only"]:
    print(f"--- {scope} ---")
    print(cmp.loc[cmp["scope"] == scope, cols].round(2).to_string(index=False), "\n")

seed 42: train 38,477 rows, last train date 2025-10-31; test drawn from 10 months, 19.8% from Sep-Oct

--- all ---
          fold                             model scope  rows     MAE  MAPE_%    RMSE    bias
          main             B0 global median rate   all  9523 1148.90   70.21 1569.00 -364.60
          main B1 median rpm by equipment x band   all  9523  140.53    6.40  638.82  -43.12
random seed 42             B0 global median rate   all  9523 1127.82   68.87 1503.68 -351.79
random seed 42 B1 median rpm by equipment x band   all  9523  145.22    6.70  542.03  -17.82
 random seed 1             B0 global median rate   all  9523 1141.97   71.32 1543.54 -346.91
 random seed 1 B1 median rpm by equipment x band   all  9523  152.28    6.96  628.40  -19.32
 random seed 2             B0 global median rate   all  9523 1132.09   69.56 1519.97 -337.63
 random seed 2 B1 median rpm by equipment x band   all  9523  140.45    6.59  563.93   -9.89 

--- clean-only ---
          fold             

In [14]:
# ===== Step 6c: does the random split's extra error come from month-to-month price levels? =====
rand42_b1 = next(r for r in random_runs if r["fold"] == "random seed 42" and r["model"].startswith("B1"))
hr = rand42_b1["holdout"]
print(validation.breakdown(hr[~hr["is_label_error"]], "month").round(2).to_string(index=False))

  month  rows  label_errors    MAE  MAPE_%   RMSE    bias
2025-01   974             0 143.32    6.35 199.18  134.22
2025-02   873             0 124.27    5.44 173.84  107.12
2025-03   944             0  91.54    4.05 130.81   14.10
2025-04   920             0  82.34    3.79 114.40    5.03
2025-05   961             0  88.61    3.94 119.75  -46.44
2025-06   941             0 128.92    5.30 173.57 -111.58
2025-07   995             0  91.93    4.09 127.03  -39.78
2025-08   934             0  87.76    3.95 123.56   38.51
2025-09   890             0  89.16    3.88 123.42   -2.26
2025-10   962             0  82.69    3.87 110.56  -15.62
